In [2]:
import pandas as pd
import numpy as np

# dit関連ライブラリのインポート
from dit import Distribution
# from dit.multivariate import pid
from dit import pid

In [3]:
import pandas as pd

path_fc_01 = "/home/horiguchi/LLM/LLM_Agents_Network/experiment/outputs_college_math_small/output_01/analysis/parsed_responses/fully_connected.csv"
df_01 = pd.read_csv(path_fc_01, sep="|")

# 最終ラウンド(例: round の最大値を求める)
max_round = df_01['round'].max()

# 例: 最終ラウンドだけ抽出
df_final = df_01[df_01['round'] == max_round].copy()

# 複数行があれば agent_id と question_number の組み合わせで groupby → 最終行をとる等
df_final_unique = df_final.groupby(['agent_id','question_number'], as_index=False).last()

# これで "agent_id, question_number" 毎に 1行に絞れる想定
df_final_unique.head()


,agent_id,question_number,network_number,round,repeat,parsed_response,correct_response,correct,bias
0,0,0,0,3,0,B,B,True,unbiased
1,0,1,0,3,0,D,D,True,unbiased
2,0,2,0,3,0,B,D,False,unbiased
3,0,3,0,3,0,A,A,True,unbiased
4,0,4,0,3,0,C,C,True,unbiased


In [4]:
response_map = {"A":0, "B":1, "C":2, "D":3, "X":4}

df_final_unique['resp_cat'] = df_final_unique['parsed_response'].map(response_map)

# 正解ラベル correct_response も同様にマップ
df_final_unique['corr_cat'] = df_final_unique['correct_response'].map(response_map)

# 正誤フラグは True/False なのでそのまま 1/0 に
df_final_unique['correct_flag'] = df_final_unique['correct'].astype(int)


In [5]:
df_final_unique

,agent_id,question_number,network_number,round,repeat,parsed_response,correct_response,correct,bias,resp_cat,corr_cat,correct_flag
0,0,0,0,3,0,B,B,True,unbiased,1,1,1
1,0,1,0,3,0,D,D,True,unbiased,3,3,1
2,0,2,0,3,0,B,D,False,unbiased,1,3,0
3,0,3,0,3,0,A,A,True,unbiased,0,0,1
4,0,4,0,3,0,C,C,True,unbiased,2,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...
245,24,5,0,3,0,D,D,True,unbiased,3,3,1
246,24,6,0,3,0,C,C,True,unbiased,2,2,1
247,24,7,0,3,0,B,C,False,unbiased,1,2,0
248,24,8,0,3,0,C,C,True,unbiased,2,2,1


In [9]:
df = df_final_unique.sample(n=10, random_state=42)
df_selected = df[
    [
        'agent_id',
        'question_number',
        # 'network_number',
        'round',
        'parsed_response',
        'correct_response',
        'correct'
    ]
]
latex_code = df_selected.to_latex(index=False)  # index=False で行番号を消す
print(latex_code)


\begin{tabular}{rrrllr}
\toprule
agent_id & question_number & round & parsed_response & correct_response & correct \\
\midrule
14 & 2 & 3 & B & D & False \\
0 & 6 & 3 & C & C & True \\
9 & 7 & 3 & C & C & True \\
6 & 0 & 3 & B & B & True \\
11 & 2 & 3 & B & D & False \\
18 & 1 & 3 & D & D & True \\
19 & 7 & 3 & B & C & False \\
18 & 4 & 3 & C & C & True \\
0 & 9 & 3 & C & A & False \\
10 & 4 & 3 & C & C & True \\
\bottomrule
\end{tabular}



In [81]:
import pandas as pd
from collections import Counter
from dit import Distribution

def distribution_from_data(data, rv_names=None):
    """
    離散値のタプルを多数含むリスト `data` から頻度を数え，
    outcome→確率 の辞書を作って dit.Distribution を返すヘルパー関数。

    Parameters
    ----------
    data : list of tuple
        例: [(0,1,0), (0,1,1), (1,1,0), ...] のように
        (X1, X2, ..., Y) を格納したリスト
    rv_names : list of str, optional
        変数名を指定する場合に使う。

    Returns
    -------
    dist : dit.Distribution
        data中のユニークなタプルをoutcomeとして持つ確率分布
    """
    counter = Counter(data)
    total = sum(counter.values())

    pmf_dict = {}
    for outcome, count in counter.items():
        pmf_dict[outcome] = count / total

    dist = Distribution(pmf_dict)
    if rv_names is not None:
        dist.set_rv_names(rv_names)

    return dist


In [83]:
from dit import pid

def compute_pid_2sources(X1, X2, Y):
    """
    2つのソース変数 (X1, X2) とターゲット変数 Y を持つデータに対し
    PIDを計算して冗長性 R, 相乗性 S, ユニーク情報 U を返す例。

    Parameters
    ----------
    X1, X2, Y : array-like (同じ長さ)
        離散値のリスト (例: [0,1,2, ...])。

    Returns
    -------
    pid_result : dict
        {'R': ..., 'U(X1)': ..., 'U(X2)': ..., 'S': ..., 'I_total': ...} など
    """
    # (X1, X2, Y) をまとめてタプルに
    data = list(zip(X1, X2, Y))

    # ディストリビューション生成
    dist = distribution_from_data(data, rv_names=['X1','X2','Y'])

    # PID計算 (cmethod='MIN' など他の手法もあり)
    result = pid(dist, cmethod='MIN')

    # 相互情報量 I(X1,X2; Y) (全体の大きさ) も参考にしたいので計算
    I_total = dist.information(['X1','X2'], ['Y'])

    return {
        'R': result['R'],
        'U(X1)': result['U(X1)'],
        'U(X2)': result['U(X2)'],
        'S': result['S'],
        'I_total': I_total
    }


In [84]:
def compute_pid_for_two_agents(df_final_unique, agent_i, agent_j):
    """
    df_final_unique から agent_i, agent_j のデータを取り出し，
    2ソースPIDを計算。

    Return:
      {'R': ..., 'U(X1)':..., 'U(X2)':..., 'S':..., 'I_total':...}
    """
    # agent_i のデータ
    df_i = df_final_unique[df_final_unique['agent_id'] == agent_i]
    # agent_j のデータ
    df_j = df_final_unique[df_final_unique['agent_id'] == agent_j]

    # question_number でマージ → 同じ問題に回答している行をペアに
    df_ij = pd.merge(
        df_i, df_j,
        on='question_number',
        suffixes=('_i','_j')
    )

    # X1 = agent_i の回答, X2 = agent_j の回答
    # ターゲット Y = agent_i側の correct_flag (or _j でも同じ問題なら同じはず)
    X1 = df_ij['resp_cat_i'].values
    X2 = df_ij['resp_cat_j'].values
    Y  = df_ij['correct_flag_i'].values  # 同じ問題なら i も j も正解フラグは一緒？

    # PID計算
    pid_res = compute_pid_2sources(X1, X2, Y)
    return pid_res

# 例: agent0 と agent1 のPIDを試す
res_01 = compute_pid_for_two_agents(df_final_unique, agent_i=0, agent_j=1)
print("PID results for (agent 0, agent 1):", res_01)


TypeError: 'module' object is not callable

In [85]:
# masking_rateごとに df_final_unique_mr が用意されている想定
# たとえば0.1, 0.5, 0.9 の3種類

mask_rates = [0.1, 0.5, 0.9]
all_results = []

for mr in mask_rates:
    # df_mr = ... （masking=mrのdf_final_uniqueを読み込み or 既に変数として保持）

    # agentペアをループしてPIDを計算 (例: agent0, agent1だけ)
    pid_res = compute_pid_for_two_agents(df_mr, agent_i=0, agent_j=1)

    row = {
        'masking_rate': mr,
        'agent_i': 0,
        'agent_j': 1,
        **pid_res  # {'R':..., 'U(X1)':..., ...}
    }
    all_results.append(row)

df_all = pd.DataFrame(all_results)
print(df_all)


NameError: name 'df_mr' is not defined